In [0]:
df = (
    spark.read.csv("/Volumes/workspace/shop_flow/raw_files/olist_customers_dataset.csv", header=True, inferSchema=True)
)

df.display()

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.shop_flow_bronze")


In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("workspace.shop_flow_bronze.customers")

In [0]:
spark.sql("SHOW TABLES IN workspace.shop_flow_bronze").display()

In [0]:
spark.sql("""
          SELECT COUNT(*) AS total
          FROM workspace.shop_flow_bronze.customers
          """).display()

In [0]:
spark.sql("""
          SELECT *
          FROM workspace.shop_flow_bronze.customers
          LIMIT 10
          """).display()

In [0]:
spark.sql("DESCRIBE TABLE workspace.shop_flow_bronze.customers").display()

In [0]:
orders_df = (
    spark.read.csv(
        "/Volumes/workspace/shop_flow/raw_files/olist_orders_dataset.csv",
        header=True,
        inferSchema=True
    )
)

orders_df.display()

In [0]:
orders_df.printSchema()

In [0]:
print("Rows:", orders_df.count())
print("Columns:", len(orders_df.columns))

In [0]:
from pyspark.sql.functions import col, sum, when

orders_df.select(
    *[
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in orders_df.columns
    ]
).display()

In [0]:
from pyspark.sql.functions import col, sum, when

orders_df.select(
    *[
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in orders_df.columns
    ]
).display()

In [0]:
orders_df.groupBy("order_status").count().orderBy("count", ascending=False).display()

In [0]:
from pyspark.sql.functions import min, max

orders_df.select(
    min("order_purchase_timestamp").alias("first_order"),
    max("order_purchase_timestamp").alias("last_order")
).display()

In [0]:
from pyspark.sql.functions import col, sum, when

orders_df.groupBy("order_id").count().filter(col("count") > 1).display()

In [0]:
orders_df.filter(col("customer_id").isNull()).count()

##load orders into Bronze

In [0]:
orders_df.write.format("delta").mode("overwrite").saveAsTable("workspace.shop_flow_bronze.orders")

In [0]:
spark.sql("SHOW TABLES IN workspace.shop_flow_bronze").display()

In [0]:
from pyspark.sql.functions import col, sum, when

orders_df.filter(col("order_id").isNull()).count()

# Define the ingestion function

In [0]:
def ingest_to_bronze(table_name, file_name):

    path = f"/Volumes/workspace/shop_flow/raw_files/{file_name}"

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "true")
        .load(path)
    )

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"workspace.shop_flow_bronze.{table_name}")

    print(f"Loaded: {table_name}")
    print(f"Rows: {df.count()}")




# Define the remaining tables

In [0]:
tables = {
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv"
}

for table_name, file_name in tables.items():
    ingest_to_bronze(table_name, file_name)

In [0]:
spark.sql("SHOW TABLES IN workspace.shop_flow_bronze").display()